# Feature Engineering: Head-to-Head

Прошлые встречи команд на двух уровнях:

**Пара vs пара** (`h2h_pair_*`):
- `h2h_pair_matches` - сколько раз эти конкретные пары встречались ранее
- `h2h_pair_team1_wins` - сколько из них выиграла команда 1
- `h2h_pair_team1_winrate` - доля побед команды 1 в очных встречах
- `h2h_pair_days_since_last` - дней с последней очной встречи

**Игрок vs игрок** (`h2h_players_*`): агрегируем по всем 2×2 = 4 парам игроков между командами
- `h2h_players_matches` - суммарное число встреч между игроками команд
- `h2h_players_team1_wins` - сколько из них выиграла команда 1
- `h2h_players_team1_winrate`

Результат: `data/features/head_to_head.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

PROCESSED = Path("../../data/processed")
FEATURES = Path("../../data/features")

df = pd.read_csv(PROCESSED / "matches.csv", parse_dates=["played_at"])
df = df.sort_values("played_at").reset_index(drop=True)
print(f"Матчей: {len(df)}")

Матчей: 2549


In [2]:
def _pair_key(a, b):
    return (int(min(a, b)), int(max(a, b)))


def _pair_matchup_key(k1, k2):
    return tuple(sorted([k1, k2]))


def _player_matchup_key(a, b):
    a, b = int(a), int(b)
    return (min(a, b), max(a, b))


def compute_h2h(df):
    df = df.sort_values("played_at").reset_index(drop=True)

    pair_n = defaultdict(int)
    pair_team1_wins = defaultdict(int) 
    pair_last_date = {}

    player_n = defaultdict(int)
    player_team1_wins = defaultdict(int)

    records = []

    for _, row in df.iterrows():
        k1 = _pair_key(row["player_id_1"], row["player_id_2"])
        k2 = _pair_key(row["player_id_3"], row["player_id_4"])
        pair_key = _pair_matchup_key(k1, k2)

        first_pair = pair_key[0]
        team1_is_first = (k1 == first_pair)

        rec = {"match_id": row["match_id"]}

        n = pair_n[pair_key]
        wins_first = pair_team1_wins[pair_key]
        team1_wins = wins_first if team1_is_first else (n - wins_first)
        rec["h2h_pair_matches"] = n
        rec["h2h_pair_team1_wins"] = team1_wins
        rec["h2h_pair_team1_winrate"] = (team1_wins / n) if n > 0 else np.nan
        last = pair_last_date.get(pair_key)
        rec["h2h_pair_days_since_last"] = (row["played_at"] - last).days if last is not None else np.nan

        p_team1_wins_total = 0
        p_matches_total = 0
        for a in [row["player_id_1"], row["player_id_2"]]:
            for b in [row["player_id_3"], row["player_id_4"]]:
                pk = _player_matchup_key(a, b)
                a_is_first = (int(a) == pk[0]) 
                n_p = player_n[pk]
                w_first = player_team1_wins[pk]
                t1_w = w_first if a_is_first else (n_p - w_first)
                p_team1_wins_total += t1_w
                p_matches_total += n_p
        rec["h2h_players_matches"] = p_matches_total
        rec["h2h_players_team1_wins"] = p_team1_wins_total
        rec["h2h_players_team1_winrate"] = (p_team1_wins_total / p_matches_total) if p_matches_total > 0 else np.nan

        records.append(rec)

        t1_won = 1 if row["winner"] == "team_1" else 0
        pair_n[pair_key] += 1
        pair_team1_wins[pair_key] += t1_won if team1_is_first else (1 - t1_won)
        pair_last_date[pair_key] = row["played_at"]

        for a in [row["player_id_1"], row["player_id_2"]]:
            for b in [row["player_id_3"], row["player_id_4"]]:
                pk = _player_matchup_key(a, b)
                a_is_first = (int(a) == pk[0])
                player_n[pk] += 1
                player_team1_wins[pk] += t1_won if a_is_first else (1 - t1_won)

    return pd.DataFrame(records)

In [3]:
h2h = compute_h2h(df)
print(h2h.describe())

          match_id  h2h_pair_matches  h2h_pair_team1_wins  \
count  2549.000000       2549.000000          2549.000000   
mean   2419.975284          0.403688             0.235386   
std    2176.372145          2.021238             1.402617   
min      34.000000          0.000000             0.000000   
25%     738.000000          0.000000             0.000000   
50%    1463.000000          0.000000             0.000000   
75%    3820.000000          0.000000             0.000000   
max    7478.000000         30.000000            21.000000   

       h2h_pair_team1_winrate  h2h_pair_days_since_last  h2h_players_matches  \
count              368.000000                368.000000          2549.000000   
mean                 0.513830                 61.817935             5.845037   
std                  0.460141                 93.964757            11.893163   
min                  0.000000                  5.000000             0.000000   
25%                  0.000000                 14.0

In [4]:
has_h2h = (h2h["h2h_pair_matches"] > 0).mean()
has_player_h2h = (h2h["h2h_players_matches"] > 0).mean()
print(f"Матчей с pair-H2H: {has_h2h:.1%}")
print(f"Матчей с player-H2H: {has_player_h2h:.1%}")

Матчей с pair-H2H: 14.4%
Матчей с player-H2H: 65.8%


In [ ]:
out_path = FEATURES / "head_to_head.csv"
h2h.to_csv(out_path, index=False)

Сохранено: ../../data/features/head_to_head.csv, фич: 7
